# Advanced Pandas - GroupBy, Merge & MultiIndex
## Professional Data Analysis Techniques for Real-World Challenges

This is the **advanced level** of Pandas mastery. These techniques unlock powerful data aggregation, combination, and reshaping capabilities used by professional data scientists daily!

---

## 📚 Learning Objectives

By the end of this session, you will be able to:

### GroupBy
1. Apply the split-apply-combine workflow
2. Aggregate data with `.groupby()` and `.agg()`
3. Use aggregation functions (sum, mean, count, custom)
4. Transform data within groups
5. Filter by group conditions
6. Group by multiple columns

### Merge
7. Merge DataFrames using `.merge()` and `.join()`
8. Understand different join types (inner, left, right, outer)
9. Handle duplicate keys and naming conflicts
10. Merge on index or columns flexibly

### MultiIndex
11. Create and manipulate multi-level indices
12. Slice and access multi-indexed data
13. Pivot and unstack data structures
14. Work with hierarchical data effectively

---

## Setup

In [31]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [32]:
# Create sample datasets locally (no external URLs needed)
import random
random.seed(42)
np.random.seed(42)

# Netflix Dataset
netflix = pd.DataFrame({
    'title': ['Breaking Bad', 'The Office', 'Stranger Things', 'Friends', 'The Crown', 'Narcos'] * 10,
    'type': [random.choice(['Movie', 'TV Show']) for _ in range(60)],
    'release_year': [random.randint(2015, 2024) for _ in range(60)],
    'rating': [random.choice(['TV-MA', 'TV-14', 'PG-13']) for _ in range(60)]
})

# Superstore Dataset - structured for GroupBy examples
superstore = pd.DataFrame({
    'Region': [random.choice(['North', 'South', 'East', 'West']) for _ in range(120)],
    'Product Category': [random.choice(['Electronics', 'Furniture', 'Office Supplies']) for _ in range(120)],
    'Sales': np.random.uniform(50, 500, 120),
    'Profit': np.random.uniform(-30, 150, 120),
    'Quantity': np.random.randint(1, 20, 120)
})

# Google Play Store Dataset
googleplay = pd.DataFrame({
    'App': [f'App_{i}' for i in range(50)],
    'Category': [random.choice(['Games', 'Productivity', 'Social']) for _ in range(50)],
    'Rating': np.random.uniform(2, 5, 50)
})

print("All datasets loaded successfully!")

All datasets loaded successfully!


---

# PART A: GROUPBY OPERATIONS

## What is GroupBy?

GroupBy implements the **split-apply-combine** workflow:
1. **Split**: Partition data by group key(s)
2. **Apply**: Perform operation on each group
3. **Combine**: Merge results back together

---

## Section 1: Basic GroupBy

### 1A: Single Column GroupBy

In [33]:
# Count content types in Netflix
print("Content count by type:")
type_counts = netflix.groupby('type').size()
print(type_counts)
print(f"\nType: {type(type_counts)}")

Content count by type:
type
Movie      35
TV Show    25
dtype: int64

Type: <class 'pandas.core.series.Series'>


In [34]:
# Multiple aggregations
print("Release year stats by type:")
year_stats = netflix.groupby('type')['release_year'].agg([
    ('count', 'count'),
    ('mean', 'mean'),
    ('min', 'min'),
    ('max', 'max')
]).round(1)
print(year_stats)

Release year stats by type:
         count    mean   min   max
type                              
Movie       35  2018.8  2015  2024
TV Show     25  2021.1  2015  2024


> 💡 **Best Practice**: Always check the GroupBy object type before aggregating. Use `.agg()` for multiple functions on result.

### 1B: Multiple Column GroupBy

In [35]:
# Group by multiple columns
print("Superstore: Sales by Region and Category")
if 'Region' in superstore.columns and 'Product Category' in superstore.columns:
    sales_by_region_cat = superstore.groupby(['Region', 'Product Category'])['Sales'].agg([
        'sum',
        'mean',
        'count'
    ]).round(2)
    print(sales_by_region_cat.head(10))

Superstore: Sales by Region and Category
                             sum    mean  count
Region Product Category                        
East   Electronics       1179.78  168.54      7
       Furniture         2672.32  267.23     10
       Office Supplies    628.70  209.57      3
North  Electronics       3178.88  211.93     15
       Furniture         3892.45  324.37     12
       Office Supplies   2027.66  202.77     10
South  Electronics       3044.90  234.22     13
       Furniture         2926.76  365.84      8
       Office Supplies   3397.71  308.88     11
West   Electronics       3351.94  304.72     11


---

## Section 2: Aggregation Functions

In [36]:
# NAMED AGGREGATIONS: Create clear, meaningful result column names
# This gives your aggregation results clear, business-friendly names
if 'Sales' in superstore.columns and 'Region' in superstore.columns:
    print("Named Aggregations (Superstore by Region):")
    print("=" * 60)
    
    # Classic approach (column names become confusing)
    result_old = superstore.groupby('Region')['Sales'].agg(['sum', 'mean', 'std'])
    print("\nOld way (confusing column names):")
    print(result_old.head())
    
    # Professional approach: Using agg with list of tuples for named results
    result_named = (superstore.groupby('Region')['Sales']
                   .agg(['sum', 'mean', 'std', 'count', 'median'])
                   .rename(columns={'sum': 'Total Sales', 
                                   'mean': 'Average Sale',
                                   'std': 'Std Dev',
                                   'count': 'Count',
                                   'median': 'Median'})
                   .round(2))
    print("\n~ Better way (clear, business-friendly names): ~")
    print(result_named)


Named Aggregations (Superstore by Region):

Old way (confusing column names):
                sum        mean         std
Region                                     
East    4480.797300  224.039865  130.440125
North   9098.991443  245.918688  127.402963
South   9369.368106  292.792753  131.160162
West    8814.787625  284.347988  142.973401

~ Better way (clear, business-friendly names): ~
        Total Sales  Average Sale  Std Dev  Count  Median
Region                                                   
East        4480.80        224.04   130.44     20  181.26
North       9098.99        245.92   127.40     37  244.38
South       9369.37        292.79   131.16     32  295.12
West        8814.79        284.35   142.97     31  284.03


In [37]:
# CUSTOM AGGREGATION: Create business-specific metrics
def range_func(x):
    """Calculate range (max - min) for identifying volatility"""
    return x.max() - x.min()

def coefficient_of_variation(x):
    """Relative variability: std / mean (useful for comparing volatility across different scales)"""
    return (x.std() / x.mean() * 100) if x.mean() != 0 else 0

def percentile_90(x):
    """90th percentile: 90% of values are below this"""
    return x.quantile(0.9)

if 'Sales' in superstore.columns and 'Region' in superstore.columns:
    print("Custom Business Metrics by Region:")
    print("=" * 70)
    
    # Use aggregation functions properly
    result = superstore.groupby('Region')['Sales'].agg(
        Mean='mean',
        Median='median',
        Range=range_func,
        StdDev='std',
        CV_Percent=coefficient_of_variation,
        P90=percentile_90,
        Count='count'
    ).round(2)
    
    print(result)
    print("\n💡 Insight: CV% shows how volatile each region is. Higher = more unpredictable sales.")


Custom Business Metrics by Region:
          Mean  Median   Range  StdDev  CV_Percent     P90  Count
Region                                                           
East    224.04  181.26  432.05  130.44       58.22  381.34     20
North   245.92  244.38  409.10  127.40       51.81  423.46     37
South   292.79  295.12  410.32  131.16       44.80  452.57     32
West    284.35  284.03  423.20  142.97       50.28  472.77     31

💡 Insight: CV% shows how volatile each region is. Higher = more unpredictable sales.


---

## Section 3: Filtering Groups

In [38]:
# FILTER: Keep only groups meeting specific business criteria
print("BUSINESS SCENARIO: Identify high-performing sales regions")
print("=" * 70)

if 'Sales' in superstore.columns and 'Region' in superstore.columns:
    # Calculate total sales by region
    region_totals = superstore.groupby('Region')['Sales'].transform('sum')
    
    # Filter: Keep only regions with total sales > $50,000
    high_performing = superstore[region_totals > 50000]
    print(f"\nTotal rows: {len(superstore)}")
    print(f"Rows in high-performing regions (>$50k): {len(high_performing)}")
    print(f"Regions: {high_performing['Region'].unique().tolist()}")

# Netflix example: Find content types with many entries
print("\n" + "=" * 70)
print("NETFLIX: Content types with 100+ titles")
netflix_filtered = netflix.groupby('type').filter(lambda x: len(x) >= 100)
print(f"Original rows: {len(netflix)}")
print(f"Filtered rows: {len(netflix_filtered)}")
print(f"Content types: {netflix_filtered['type'].unique().tolist()}")
print(f"Row percentages per type:\n{netflix_filtered['type'].value_counts(normalize=True).mul(100).round(1)}")


BUSINESS SCENARIO: Identify high-performing sales regions

Total rows: 120
Rows in high-performing regions (>$50k): 0
Regions: []

NETFLIX: Content types with 100+ titles
Original rows: 60
Filtered rows: 0
Content types: []
Row percentages per type:
Series([], Name: proportion, dtype: float64)


In [39]:
# Find groups by aggregated value
if 'Sales' in superstore.columns and 'Region' in superstore.columns:
    high_sales_regions = (superstore.groupby('Region')['Sales']
                         .sum()
                         .sort_values(ascending=False))
    print("Regions by total sales:")
    print(high_sales_regions)
    
    # Get data only from top regions
    top_regions = high_sales_regions.head(2).index
    top_region_data = superstore[superstore['Region'].isin(top_regions)]
    print(f"\nRows from top 2 regions: {len(top_region_data)}")

Regions by total sales:
Region
South    9369.368106
North    9098.991443
West     8814.787625
East     4480.797300
Name: Sales, dtype: float64

Rows from top 2 regions: 69


---

## Section 4: Transform Operations

In [40]:
# TRANSFORM: Apply function to each group and return same-shape DataFrame
# Key insight: Transform preserves original shape, unlike aggregation

print("USE CASE 1: Standardize sales within each region (z-score)")
print("=" * 70)

if 'Sales' in superstore.columns and 'Region' in superstore.columns:
    def standardize(x):
        """Z-score: (value - mean) / std. Centers at 0, scale of 1."""
        return (x - x.mean()) / x.std()
    
    superstore_copy = superstore.copy()
    superstore_copy['Sales_Standardized'] = (superstore.groupby('Region')['Sales']
                                             .transform(standardize))
    
    print("Original vs Standardized (by Region):")
    sample = superstore_copy[['Region', 'Sales', 'Sales_Standardized']].head(10)
    print(sample)
    print(f"\nMean by region (should be ~0):")
    print(superstore_copy.groupby('Region')['Sales_Standardized'].mean().round(4))
    print(f"\nStd by region (should be ~1):")
    print(superstore_copy.groupby('Region')['Sales_Standardized'].std().round(4))

print("\n" + "=" * 70)
print("USE CASE 2: Rank sales within each region (who's top in their region?)")

if 'Sales' in superstore.columns and 'Region' in superstore.columns:
    superstore_copy['Sales_Rank_In_Region'] = (superstore.groupby('Region')['Sales']
                                               .transform('rank', ascending=False)
                                               .astype('int'))
    
    print("Top 3 sellers per region:")
    for region in superstore_copy['Region'].unique():
        top3 = superstore_copy[superstore_copy['Region'] == region].nsmallest(3, 'Sales_Rank_In_Region')
        print(f"\n{region}: {top3[['Region', 'Sales', 'Sales_Rank_In_Region']].to_string(index=False)}")

print("\n" + "=" * 70)
print("USE CASE 3: Calculate % of region total (market share within region)")

if 'Sales' in superstore.columns and 'Region' in superstore.columns:
    superstore_copy['Pct_of_Region'] = (superstore.groupby('Region')['Sales']
                                        .transform(lambda x: x / x.sum() * 100))
    
    print("Sample (% of regional sales):")
    print(superstore_copy.groupby('Region')['Pct_of_Region'].agg(['sum', 'mean', 'max']).round(2))


USE CASE 1: Standardize sales within each region (z-score)
Original vs Standardized (by Region):
  Region       Sales  Sales_Standardized
0   East  218.543053           -0.042140
1   West  477.821438            1.353213
2   East  379.397274            1.191025
3   West  319.396318            0.245139
4   West  120.208388           -1.148043
5  North  120.197534           -0.986799
6  South   76.137625           -1.651836
7  South  439.779266            1.120664
8  North  320.501755            0.585411
9   East  368.632660            1.108499

Mean by region (should be ~0):
Region
East     0.0
North    0.0
South    0.0
West    -0.0
Name: Sales_Standardized, dtype: float64

Std by region (should be ~1):
Region
East     1.0
North    1.0
South    1.0
West     1.0
Name: Sales_Standardized, dtype: float64

USE CASE 2: Rank sales within each region (who's top in their region?)
Top 3 sellers per region:

East: Region      Sales  Sales_Rank_In_Region
  East 484.534415                     1
  Ea

> 🔍 **Real-World Use Case**: Transform is essential for creating features like "sales relative to region average" or "normalized metrics per group."

---

# PART B: MERGE OPERATIONS 

## What is Merge?

Merge combines two DataFrames based on common columns or indices. It's like SQL joins.

---

## Section 5: Different Join Types

In [41]:
# Create realistic business datasets: Orders and Returns
print("BUSINESS SCENARIO: Analyze orders and returns for customer service insights")
print("=" * 70)

# Orders dataset
orders = pd.DataFrame({
    'OrderID': [1001, 1002, 1003, 1004, 1005],
    'CustomerID': [101, 102, 101, 103, 104],  # Note: Customer 105 has no orders
    'OrderAmount': [150, 320, 210, 450, 89],
    'OrderDate': ['2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19']
})

# Returns dataset (some orders are returned, 1007 is unmatched)
returns = pd.DataFrame({
    'ReturnID': [2001, 2002, 2003],
    'OrderID': [1002, 1003, 1007],  # 1007 doesn't exist in orders
    'ReturnAmount': [320, 100, 75],
    'ReturnDate': ['2024-01-20', '2024-01-21', '2024-01-22']
})

# Customers (not all have orders)
customers = pd.DataFrame({
    'CustomerID': [101, 102, 103, 104, 105],
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'Segment': ['Premium', 'Standard', 'Premium', 'Standard', 'VIP']
})

print("Orders DataFrame:")
print(orders)
print("\nReturns DataFrame:")
print(returns)
print("\nCustomers DataFrame:")
print(customers)


BUSINESS SCENARIO: Analyze orders and returns for customer service insights
Orders DataFrame:
   OrderID  CustomerID  OrderAmount   OrderDate
0     1001         101          150  2024-01-15
1     1002         102          320  2024-01-16
2     1003         101          210  2024-01-17
3     1004         103          450  2024-01-18
4     1005         104           89  2024-01-19

Returns DataFrame:
   ReturnID  OrderID  ReturnAmount  ReturnDate
0      2001     1002           320  2024-01-20
1      2002     1003           100  2024-01-21
2      2003     1007            75  2024-01-22

Customers DataFrame:
   CustomerID     Name   Segment
0         101    Alice   Premium
1         102      Bob  Standard
2         103  Charlie   Premium
3         104    David  Standard
4         105      Eve       VIP


In [42]:
# INNER JOIN: Only records that exist in BOTH DataFrames
print("\n🔍 INNER JOIN: Matching records only (Orders with Returns)")
print("=" * 70)
print("Use when: You want ONLY orders that had returns (quality control analysis)")

inner_merge = pd.merge(orders, returns, on='OrderID', how='inner')
print("\nOrders that were RETURNED:")
print(inner_merge)
print(f"\n✓ Result: {len(inner_merge)} rows (only matched OrderIDs: {inner_merge['OrderID'].tolist()})")
print(f"  Lost: {len(orders) - len(inner_merge)} orders from original")

# Calculate impact
total_returned = inner_merge['ReturnAmount'].sum()
total_ordered = orders['OrderAmount'].sum()
print(f"\n💰 Business insight: ${total_returned} returned out of ${total_ordered} ordered = {total_returned/total_ordered*100:.1f}% return rate")



🔍 INNER JOIN: Matching records only (Orders with Returns)
Use when: You want ONLY orders that had returns (quality control analysis)

Orders that were RETURNED:
   OrderID  CustomerID  OrderAmount   OrderDate  ReturnID  ReturnAmount  \
0     1002         102          320  2024-01-16      2001           320   
1     1003         101          210  2024-01-17      2002           100   

   ReturnDate  
0  2024-01-20  
1  2024-01-21  

✓ Result: 2 rows (only matched OrderIDs: [1002, 1003])
  Lost: 3 orders from original

💰 Business insight: $420 returned out of $1219 ordered = 34.5% return rate


In [43]:
# LEFT JOIN: All records from LEFT DataFrame + matching from right
print("\n🔍 LEFT JOIN: All orders + their returns (if any)")
print("=" * 70)
print("Use when: You want ALL orders, and to identify which ones have NO returns")

left_merge = pd.merge(orders, returns, on='OrderID', how='left')
print("\nAll orders (with return info if it exists):")
print(left_merge)

print(f"\n✓ Result: {len(left_merge)} rows (all {len(orders)} orders included)")
print(f"\nOrders with NO returns (NaN in ReturnID):")
no_returns = left_merge[left_merge['ReturnID'].isna()]
print(f"  {len(no_returns)} orders: OrderIDs {no_returns['OrderID'].tolist()}")

successful_orders = no_returns['OrderAmount'].sum()
print(f"\n💰 Revenue from successful orders (no returns): ${successful_orders}")



🔍 LEFT JOIN: All orders + their returns (if any)
Use when: You want ALL orders, and to identify which ones have NO returns

All orders (with return info if it exists):
   OrderID  CustomerID  OrderAmount   OrderDate  ReturnID  ReturnAmount  \
0     1001         101          150  2024-01-15       NaN           NaN   
1     1002         102          320  2024-01-16    2001.0         320.0   
2     1003         101          210  2024-01-17    2002.0         100.0   
3     1004         103          450  2024-01-18       NaN           NaN   
4     1005         104           89  2024-01-19       NaN           NaN   

   ReturnDate  
0         NaN  
1  2024-01-20  
2  2024-01-21  
3         NaN  
4         NaN  

✓ Result: 5 rows (all 5 orders included)

Orders with NO returns (NaN in ReturnID):
  3 orders: OrderIDs [1001, 1004, 1005]

💰 Revenue from successful orders (no returns): $689


In [44]:
# RIGHT JOIN: All records from RIGHT DataFrame + matching from left
print("\n🔍 RIGHT JOIN: All returns + their order info (if it exists)")
print("=" * 70)
print("Use when: You have a transaction log (returns) and want to track which relate to original orders")

right_merge = pd.merge(orders, returns, on='OrderID', how='right')
print("\nAll returns (with order info if it exists):")
print(right_merge)

print(f"\n✓ Result: {len(right_merge)} rows (all {len(returns)} returns included)")

# Identify unmatched returns (data quality issue!)
unmatched_returns = right_merge[right_merge['OrderID'].isna()]
if len(unmatched_returns) > 0:
    print(f"\n⚠️ DATA QUALITY ALERT: {len(unmatched_returns)} returns with NO matching order!")
    print(f"   Return IDs: {unmatched_returns['ReturnID'].tolist()}")
    print(f"   Orphaned refund amount: ${unmatched_returns['ReturnAmount'].sum()}")
else:
    print(f"\n✓ All returns match to an order. Data is clean!")



🔍 RIGHT JOIN: All returns + their order info (if it exists)
Use when: You have a transaction log (returns) and want to track which relate to original orders

All returns (with order info if it exists):
   OrderID  CustomerID  OrderAmount   OrderDate  ReturnID  ReturnAmount  \
0     1002       102.0        320.0  2024-01-16      2001           320   
1     1003       101.0        210.0  2024-01-17      2002           100   
2     1007         NaN          NaN         NaN      2003            75   

   ReturnDate  
0  2024-01-20  
1  2024-01-21  
2  2024-01-22  

✓ Result: 3 rows (all 3 returns included)

✓ All returns match to an order. Data is clean!


In [45]:
# OUTER JOIN: All records from BOTH DataFrames
print("\n🔍 OUTER JOIN: Complete picture (all orders AND all returns)")
print("=" * 70)
print("Use when: You need a complete view for reconciliation and data quality checks")

outer_merge = pd.merge(orders, returns, on='OrderID', how='outer')
print("\nComplete order/return picture:")
print(outer_merge)

print(f"\n✓ Result: {len(outer_merge)} rows (all {len(orders)} orders + {len(returns)} returns)")

# Identify anomalies
print("\n--- Data Quality Analysis ---")

# Orders with no returns
only_orders = outer_merge[outer_merge['ReturnID'].isna()]
print(f"\nOrders with NO returns: {len(only_orders)}")
print(f"  OrderIDs: {only_orders['OrderID'].tolist()}")

# Returns with no matching order
only_returns = outer_merge[outer_merge['OrderID'].isna()]
print(f"\nReturns with NO matching order: {len(only_returns)}")
if len(only_returns) > 0:
    print(f"  ReturnIDs: {only_returns['ReturnID'].tolist()}")
    print(f"  ⚠️ Orphaned refund: ${only_returns['ReturnAmount'].sum()}")

# Summary
print(f"\n📊 Summary:")
print(f"  • Successful orders (no returns): {len(only_orders)}")
print(f"  • Orders with returns: {len(outer_merge) - len(only_orders) - len(only_returns)}")
print(f"  • Anomalies (unmatched): {len(only_returns)}")



🔍 OUTER JOIN: Complete picture (all orders AND all returns)
Use when: You need a complete view for reconciliation and data quality checks

Complete order/return picture:
   OrderID  CustomerID  OrderAmount   OrderDate  ReturnID  ReturnAmount  \
0     1001       101.0        150.0  2024-01-15       NaN           NaN   
1     1002       102.0        320.0  2024-01-16    2001.0         320.0   
2     1003       101.0        210.0  2024-01-17    2002.0         100.0   
3     1004       103.0        450.0  2024-01-18       NaN           NaN   
4     1005       104.0         89.0  2024-01-19       NaN           NaN   
5     1007         NaN          NaN         NaN    2003.0          75.0   

   ReturnDate  
0         NaN  
1  2024-01-20  
2  2024-01-21  
3         NaN  
4         NaN  
5  2024-01-22  

✓ Result: 6 rows (all 5 orders + 3 returns)

--- Data Quality Analysis ---

Orders with NO returns: 3
  OrderIDs: [1001, 1004, 1005]

Returns with NO matching order: 0

📊 Summary:
  • Succes

### Join Type Comparison

| Type | Behavior |
|------|----------|
| **inner** | Keys present in BOTH DataFrames |
| **left** | All keys from LEFT DataFrame |
| **right** | All keys from RIGHT DataFrame |
| **outer** | All keys from BOTH DataFrames |

---

## Section 6: Advanced Merge Scenarios

In [46]:
# SCENARIO: Merge on different column names
# Orders use 'CustomerID', Customers use 'CustID' (inconsistent naming - common in real data!)

print("SCENARIO: Keys have different names")
print("=" * 70)

orders_sample = orders.head(3)
customers_renamed = customers.rename(columns={'CustomerID': 'CustID'})

print("Orders (uses 'CustomerID'):")
print(orders_sample[['OrderID', 'CustomerID', 'OrderAmount']])

print("\nCustomers (uses 'CustID'):")
print(customers_renamed[['CustID', 'Name', 'Segment']])

print("\nMerge on different key names:")
merged_diff_keys = pd.merge(
    orders_sample, 
    customers_renamed, 
    left_on='CustomerID',  # Column name in LEFT DataFrame
    right_on='CustID',      # Column name in RIGHT DataFrame
    how='inner'
)
print(merged_diff_keys)
print("\n✓ Now you can analyze orders with customer information!")


SCENARIO: Keys have different names
Orders (uses 'CustomerID'):
   OrderID  CustomerID  OrderAmount
0     1001         101          150
1     1002         102          320
2     1003         101          210

Customers (uses 'CustID'):
   CustID     Name   Segment
0     101    Alice   Premium
1     102      Bob  Standard
2     103  Charlie   Premium
3     104    David  Standard
4     105      Eve       VIP

Merge on different key names:
   OrderID  CustomerID  OrderAmount   OrderDate  CustID   Name   Segment
0     1001         101          150  2024-01-15     101  Alice   Premium
1     1002         102          320  2024-01-16     102    Bob  Standard
2     1003         101          210  2024-01-17     101  Alice   Premium

✓ Now you can analyze orders with customer information!


In [47]:
# SCENARIO: Merge on index instead of columns
# Useful when indices are the natural keys (like product codes, customer IDs as index)

print("SCENARIO: Merge on indices")
print("=" * 70)

# Set CustomerID as index (more efficient for repeated lookups)
customers_indexed = customers.set_index('CustomerID')
orders_indexed = orders.set_index('CustomerID')

print("Orders (indexed by CustomerID):")
print(orders_indexed[['OrderID', 'OrderAmount']].head(3))

print("\nCustomers (indexed by CustomerID):")
print(customers_indexed[['Name', 'Segment']].head(3))

print("\nMerge on indices:")
merged_on_index = pd.merge(
    orders_indexed, 
    customers_indexed, 
    left_index=True,   # Use left index as key
    right_index=True,  # Use right index as key
    how='inner'
)
print(merged_on_index)
print("\n💡 Index merges are FAST. Use when indices are meaningful keys.")


SCENARIO: Merge on indices
Orders (indexed by CustomerID):
            OrderID  OrderAmount
CustomerID                      
101            1001          150
102            1002          320
101            1003          210

Customers (indexed by CustomerID):
               Name   Segment
CustomerID                   
101           Alice   Premium
102             Bob  Standard
103         Charlie   Premium

Merge on indices:
            OrderID  OrderAmount   OrderDate     Name   Segment
CustomerID                                                     
101            1001          150  2024-01-15    Alice   Premium
102            1002          320  2024-01-16      Bob  Standard
101            1003          210  2024-01-17    Alice   Premium
103            1004          450  2024-01-18  Charlie   Premium
104            1005           89  2024-01-19    David  Standard

💡 Index merges are FAST. Use when indices are meaningful keys.


In [48]:
# SCENARIO: Duplicate column names requiring suffixes
# Real-world example: Orders and Shipments both have 'Date' and 'Status'

print("SCENARIO: Duplicate column names (Order vs Shipment tracking)")
print("=" * 70)

order_data = pd.DataFrame({
    'OrderID': [1001, 1002, 1003],
    'Date': ['2024-01-15', '2024-01-16', '2024-01-17'],  # Order date
    'Status': ['Confirmed', 'Pending', 'Confirmed'],       # Order status
    'Amount': [150, 320, 210]
})

shipment_data = pd.DataFrame({
    'OrderID': [1001, 1002, 1003],
    'Date': ['2024-01-17', '2024-01-18', '2024-01-20'],  # Ship date
    'Status': ['Shipped', 'Processing', 'Shipped'],       # Shipment status
    'TrackingNum': ['TRACK001', 'TRACK002', 'TRACK003']
})

print("Order data (has Date and Status):")
print(order_data)
print("\nShipment data (also has Date and Status):")
print(shipment_data)

print("\n❌ WITHOUT suffixes - WILL CAUSE ERROR or confusion:")
# Uncommenting this would show: ValueError: columns overlap but no suffix specified
# pd.merge(order_data, shipment_data, on='OrderID', how='inner')

print("\n✓ WITH suffixes - Each column gets clear context:")
merged_with_suffix = pd.merge(
    order_data, 
    shipment_data, 
    on='OrderID', 
    how='inner',
    suffixes=('_order', '_ship')  # Clear labels for disambiguation
)
print(merged_with_suffix)

print("\nNow you can see BOTH dates and statuses:")
print(merged_with_suffix[['OrderID', 'Date_order', 'Date_ship', 'Status_order', 'Status_ship']])
print("\n💡 Best Practice: Use descriptive suffixes that show WHERE the data came from.")


SCENARIO: Duplicate column names (Order vs Shipment tracking)
Order data (has Date and Status):
   OrderID        Date     Status  Amount
0     1001  2024-01-15  Confirmed     150
1     1002  2024-01-16    Pending     320
2     1003  2024-01-17  Confirmed     210

Shipment data (also has Date and Status):
   OrderID        Date      Status TrackingNum
0     1001  2024-01-17     Shipped    TRACK001
1     1002  2024-01-18  Processing    TRACK002
2     1003  2024-01-20     Shipped    TRACK003

❌ WITHOUT suffixes - WILL CAUSE ERROR or confusion:

✓ WITH suffixes - Each column gets clear context:
   OrderID  Date_order Status_order  Amount   Date_ship Status_ship  \
0     1001  2024-01-15    Confirmed     150  2024-01-17     Shipped   
1     1002  2024-01-16      Pending     320  2024-01-18  Processing   
2     1003  2024-01-17    Confirmed     210  2024-01-20     Shipped   

  TrackingNum  
0    TRACK001  
1    TRACK002  
2    TRACK003  

Now you can see BOTH dates and statuses:
   OrderID

> ⚠️ **Common Pitfall**: When merging creates duplicate column names, specify `suffixes` to distinguish them. Otherwise, you'll get an error or unexpected results.

---

# PART C: MULTIINDEX OPERATIONS (Session 21)

## What is MultiIndex?

A MultiIndex (hierarchical index) has **multiple levels** of indexing, enabling analysis of higher-dimensional data.

---

## Section 7: Creating MultiIndex

In [49]:
# Create realistic hierarchical business data
print("BUSINESS DATA: Sales by Region, Category, and Quarter")
print("=" * 70)

sales_data = pd.DataFrame({
    'Region': ['North', 'North', 'North', 'North', 'North', 'North',
               'South', 'South', 'South', 'South', 'South', 'South',
               'East', 'East', 'East', 'East', 'East', 'East'],
    'Category': ['Electronics', 'Electronics', 'Clothing', 'Clothing', 'Food', 'Food'] * 3,
    'Quarter': ['Q1', 'Q2', 'Q1', 'Q2', 'Q1', 'Q2'] * 3,
    'Sales': [45000, 52000, 12000, 15000, 8000, 9000,
              38000, 41000, 18000, 22000, 11000, 13000,
              55000, 58000, 14000, 16000, 9000, 10000],
    'Profit': [9000, 10400, 2400, 3000, 1600, 1800,
               7600, 8200, 3600, 4400, 2200, 2600,
               11000, 11600, 2800, 3200, 1800, 2000]
})

print("Original flat data structure:")
print(sales_data)
print(f"\nShape: {sales_data.shape}")

BUSINESS DATA: Sales by Region, Category, and Quarter
Original flat data structure:
   Region     Category Quarter  Sales  Profit
0   North  Electronics      Q1  45000    9000
1   North  Electronics      Q2  52000   10400
2   North     Clothing      Q1  12000    2400
3   North     Clothing      Q2  15000    3000
4   North         Food      Q1   8000    1600
5   North         Food      Q2   9000    1800
6   South  Electronics      Q1  38000    7600
7   South  Electronics      Q2  41000    8200
8   South     Clothing      Q1  18000    3600
9   South     Clothing      Q2  22000    4400
10  South         Food      Q1  11000    2200
11  South         Food      Q2  13000    2600
12   East  Electronics      Q1  55000   11000
13   East  Electronics      Q2  58000   11600
14   East     Clothing      Q1  14000    2800
15   East     Clothing      Q2  16000    3200
16   East         Food      Q1   9000    1800
17   East         Food      Q2  10000    2000

Shape: (18, 5)


In [50]:
# Create MultiIndex from flat data
print("Creating MultiIndex with 3 levels: Region → Category → Quarter")
print("=" * 70)

# Method 1: set_index() with multiple columns (creates hierarchical index)
multi_indexed = sales_data.set_index(['Region', 'Category', 'Quarter'])

print("\nFlat structure (before):")
print(f"Index type: Simple integer index")
print(f"Columns: {sales_data.columns.tolist()}")

print("\nHierarchical structure (after):")
print(f"Index type: {type(multi_indexed.index)}")
print(f"Index levels: {multi_indexed.index.nlevels}")
print(f"Index names: {multi_indexed.index.names}")
print(f"Number of rows: {len(multi_indexed)}")

print("\nMultiIndexed data:")
print(multi_indexed)

print("\n💡 Benefits of MultiIndex:")
print("   • Natural grouping hierarchy (Region contains Categories contains Quarters)")
print("   • Fast lookups: sales.loc['North'] gets entire region instantly")
print("   • Easy slicing: sales.xs('Electronics', level=1) gets all Electronics")
print("   • Familiar for hierarchical data (org structures, time series, geographies)")

Creating MultiIndex with 3 levels: Region → Category → Quarter

Flat structure (before):
Index type: Simple integer index
Columns: ['Region', 'Category', 'Quarter', 'Sales', 'Profit']

Hierarchical structure (after):
Index type: <class 'pandas.core.indexes.multi.MultiIndex'>
Index levels: 3
Index names: ['Region', 'Category', 'Quarter']
Number of rows: 18

MultiIndexed data:
                            Sales  Profit
Region Category    Quarter               
North  Electronics Q1       45000    9000
                   Q2       52000   10400
       Clothing    Q1       12000    2400
                   Q2       15000    3000
       Food        Q1        8000    1600
                   Q2        9000    1800
South  Electronics Q1       38000    7600
                   Q2       41000    8200
       Clothing    Q1       18000    3600
                   Q2       22000    4400
       Food        Q1       11000    2200
                   Q2       13000    2600
East   Electronics Q1       55000 

In [51]:
# Method 2: from_product() - Create Cartesian product (all combinations)
print("\nMethod 2: from_product() - Create all possible combinations")
print("=" * 70)

regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Food']
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

# Creates ALL combinations: 4 regions × 3 categories × 4 quarters = 48 rows
combinations = pd.MultiIndex.from_product(
    [regions, categories, quarters],
    names=['Region', 'Category', 'Quarter']
)

combo_df = pd.DataFrame(
    {'TargetSales': 50000},  # Template with default values
    index=combinations
)

print(f"All possible combinations: {len(regions)} × {len(categories)} × {len(quarters)} = {len(combo_df)} rows")
print("\nFirst 12 combinations (template structure):")
print(combo_df.head(12))

print("\n💡 Use from_product() when:")
print("   • You need a template with ALL expected combinations")
print("   • You want to compare actual vs expected (fill in missing data)")
print("   • Creating forecasts or budgets for every group")


Method 2: from_product() - Create all possible combinations
All possible combinations: 4 × 3 × 4 = 48 rows

First 12 combinations (template structure):
                            TargetSales
Region Category    Quarter             
North  Electronics Q1             50000
                   Q2             50000
                   Q3             50000
                   Q4             50000
       Clothing    Q1             50000
                   Q2             50000
                   Q3             50000
                   Q4             50000
       Food        Q1             50000
                   Q2             50000
                   Q3             50000
                   Q4             50000

💡 Use from_product() when:
   • You need a template with ALL expected combinations
   • You want to compare actual vs expected (fill in missing data)
   • Creating forecasts or budgets for every group


---

## Section 8: Slicing MultiIndex Data

In [52]:
# Slice at different levels
print("MultiIndex data:")
print(multi_indexed)

print("\nAll data for 'North' region:")
print(multi_indexed.loc['North'])

print("\nData for North, Electronics quarter Q1:")
print(multi_indexed.loc[('North', 'Electronics', 'Q1')])

MultiIndex data:
                            Sales  Profit
Region Category    Quarter               
North  Electronics Q1       45000    9000
                   Q2       52000   10400
       Clothing    Q1       12000    2400
                   Q2       15000    3000
       Food        Q1        8000    1600
                   Q2        9000    1800
South  Electronics Q1       38000    7600
                   Q2       41000    8200
       Clothing    Q1       18000    3600
                   Q2       22000    4400
       Food        Q1       11000    2200
                   Q2       13000    2600
East   Electronics Q1       55000   11000
                   Q2       58000   11600
       Clothing    Q1       14000    2800
                   Q2       16000    3200
       Food        Q1        9000    1800
                   Q2       10000    2000

All data for 'North' region:
                     Sales  Profit
Category    Quarter               
Electronics Q1       45000    9000
        

In [53]:
# Cross-section (xs) - Professional way to slice MultiIndex
print("SLICING TECHNIQUES: Getting specific parts of MultiIndex")
print("=" * 70)

print("\nTechnique 1: .loc[] - Simple slicing at first level")
print("~ Get all data for 'North' region ~")
north_data = multi_indexed.loc['North']
print(north_data)
print(f"Result shape: {north_data.shape} (still has Category, Quarter in index)")

print("\n" + "=" * 70)
print("\nTechnique 2: .xs() - Cross-section at ANY level")
print("~ Get ALL 'Electronics' sales across all regions and quarters ~")
electronics = multi_indexed.xs('Electronics', level='Category')
print(electronics)
print(f"Result shape: {electronics.shape}")

print("\n" + "=" * 70)
print("\nTechnique 3: .xs() - Multiple levels")
print("~ Get 'Electronics' in 'North' ~")
north_electronics = multi_indexed.xs(('North', 'Electronics'), level=('Region', 'Category'))
print(north_electronics)
print(f"Result shape: {north_electronics.shape} (only Quarter in index)")

print("\n" + "=" * 70)
print("\nTechnique 4: .loc[] with tuple - Specific cell")
print("~ Get 'North' → 'Electronics' → 'Q1' sales value ~")
specific_sale = multi_indexed.loc[('North', 'Electronics', 'Q1'), 'Sales']
print(f"Value: {specific_sale} (single value, not DataFrame)")

print("\n" + "=" * 70)
print("\n💡 When to use each:")
print("   .loc['North']: Simple, single-level access")
print("   .xs('Electronics', level=1): Access any level (don't need previous levels)")
print("   .loc[('North', 'Electronics', 'Q1')]: Need specific nested value")

SLICING TECHNIQUES: Getting specific parts of MultiIndex

Technique 1: .loc[] - Simple slicing at first level
~ Get all data for 'North' region ~
                     Sales  Profit
Category    Quarter               
Electronics Q1       45000    9000
            Q2       52000   10400
Clothing    Q1       12000    2400
            Q2       15000    3000
Food        Q1        8000    1600
            Q2        9000    1800
Result shape: (6, 2) (still has Category, Quarter in index)


Technique 2: .xs() - Cross-section at ANY level
~ Get ALL 'Electronics' sales across all regions and quarters ~
                Sales  Profit
Region Quarter               
North  Q1       45000    9000
       Q2       52000   10400
South  Q1       38000    7600
       Q2       41000    8200
East   Q1       55000   11000
       Q2       58000   11600
Result shape: (6, 2)


Technique 3: .xs() - Multiple levels
~ Get 'Electronics' in 'North' ~
         Sales  Profit
Quarter               
Q1       45000    900

---

## Section 9: Pivot & Stack/Unstack

In [54]:
# UNSTACK: Convert MultiIndex to wide format (useful for comparison/pivoting)
print("UNSTACK: Convert long format → wide format")
print("=" * 70)

print("\nOriginal (long format - one row per combination):")
print(multi_indexed)

print("\n--- UNSTACK LEVEL 2 (Quarter) ---")
print("Convert Quarter from index to columns")
wide_quarter = multi_indexed['Sales'].unstack(level='Quarter')
print(wide_quarter)
print(f"Shape: {wide_quarter.shape}")
print("\n💡 Now easy to see: Each row is a Region-Category pair, columns are Quarters")
print("   Perfect for: Comparing sales trends across quarters for each product line")

print("\n" + "=" * 70)
print("\n--- UNSTACK LEVEL 1 (Category) ---")
print("Convert Category from index to columns")
wide_category = multi_indexed['Sales'].unstack(level='Category')
print(wide_category)
print("\n💡 Now easy to see: Each row is a Region-Quarter pair, columns are Categories")
print("   Perfect for: Comparing performance of different product categories")

print("\n" + "=" * 70)
print("\n--- UNSTACK ALL LEVELS ---")
print("Convert all index levels to columns")
wide_all = multi_indexed['Sales'].unstack(level=[1, 2])  # Category and Quarter
print(wide_all)
print(f"Shape: {wide_all.shape}")
print("\n💡 Now each row is just Region, columns are all (Category, Quarter) combinations")

UNSTACK: Convert long format → wide format

Original (long format - one row per combination):
                            Sales  Profit
Region Category    Quarter               
North  Electronics Q1       45000    9000
                   Q2       52000   10400
       Clothing    Q1       12000    2400
                   Q2       15000    3000
       Food        Q1        8000    1600
                   Q2        9000    1800
South  Electronics Q1       38000    7600
                   Q2       41000    8200
       Clothing    Q1       18000    3600
                   Q2       22000    4400
       Food        Q1       11000    2200
                   Q2       13000    2600
East   Electronics Q1       55000   11000
                   Q2       58000   11600
       Clothing    Q1       14000    2800
                   Q2       16000    3200
       Food        Q1        9000    1800
                   Q2       10000    2000

--- UNSTACK LEVEL 2 (Quarter) ---
Convert Quarter from index to c

In [55]:
# STACK: Convert wide format → long format (reverse of unstack)
print("STACK: Convert wide format → long format")
print("=" * 70)

# Take the wide quarter data and stack it back
print("\nStarting with wide format (Quarters as columns):")
print(wide_quarter)

print("\nAfter stacking (Quarter goes back to index):")
stacked = wide_quarter.stack()
print(stacked.head(10))
print(f"Type: {type(stacked)}")
print(f"Index levels: {stacked.index.nlevels}")

print("\n💡 Use stack() when:")
print("   • Wide data needs to be reshaped to long format")
print("   • Preparing data for analysis tools that expect long format")
print("   • Combining/comparing multiple columns with same meaning")

STACK: Convert wide format → long format

Starting with wide format (Quarters as columns):
Quarter                Q1     Q2
Region Category                 
East   Clothing     14000  16000
       Electronics  55000  58000
       Food          9000  10000
North  Clothing     12000  15000
       Electronics  45000  52000
       Food          8000   9000
South  Clothing     18000  22000
       Electronics  38000  41000
       Food         11000  13000

After stacking (Quarter goes back to index):
Region  Category     Quarter
East    Clothing     Q1         14000
                     Q2         16000
        Electronics  Q1         55000
                     Q2         58000
        Food         Q1          9000
                     Q2         10000
North   Clothing     Q1         12000
                     Q2         15000
        Electronics  Q1         45000
                     Q2         52000
dtype: int64
Type: <class 'pandas.core.series.Series'>
Index levels: 3

💡 Use stack() when:

In [56]:
# PIVOT: Reshape from flat data (doesn't require MultiIndex)
print("PIVOT: Reshape flat data from long → wide format")
print("=" * 70)

print("\nOriginal flat data (flat structure, no MultiIndex):")
print(sales_data.head(9))

print("\n--- PIVOT: Regions as rows, Quarters as columns ---")
print("Use pivot() when data is in flat (non-hierarchical) format")
pivot_table = sales_data.pivot_table(
    values='Sales',
    index='Region',
    columns='Quarter',
    aggfunc='sum'  # If multiple rows per combination, sum them
)
print(pivot_table)

print("\n" + "=" * 70)
print("\nCOMPARISON: pivot_table() vs unstack()")
print("-" * 70)
print("UNSTACK:")
print("  • Requires MultiIndex")
print("  • Always takes existing hierarchical structure")
print("  • Fast, simple transformation")

print("\nPIVOT_TABLE:")
print("  • Works with flat data")
print("  • Can specify aggregation function (sum, mean, count)")
print("  • Handles multiple rows per combination automatically")
print("  • More flexible for complex reshaping")

PIVOT: Reshape flat data from long → wide format

Original flat data (flat structure, no MultiIndex):
  Region     Category Quarter  Sales  Profit
0  North  Electronics      Q1  45000    9000
1  North  Electronics      Q2  52000   10400
2  North     Clothing      Q1  12000    2400
3  North     Clothing      Q2  15000    3000
4  North         Food      Q1   8000    1600
5  North         Food      Q2   9000    1800
6  South  Electronics      Q1  38000    7600
7  South  Electronics      Q2  41000    8200
8  South     Clothing      Q1  18000    3600

--- PIVOT: Regions as rows, Quarters as columns ---
Use pivot() when data is in flat (non-hierarchical) format
Quarter     Q1     Q2
Region               
East     78000  84000
North    65000  76000
South    67000  76000


COMPARISON: pivot_table() vs unstack()
----------------------------------------------------------------------
UNSTACK:
  • Requires MultiIndex
  • Always takes existing hierarchical structure
  • Fast, simple transformation


In [57]:
# Multi-level pivot for complex analysis
print("\nMULTI-LEVEL PIVOT: Complex reshaping for business dashboards")
print("=" * 70)

print("\n--- Dashboard View: Profit by Region & Category, quarters as columns ---")
multi_pivot = sales_data.pivot_table(
    values='Profit',
    index=['Region', 'Category'],  # Hierarchical rows
    columns='Quarter',               # Columns
    aggfunc='sum'
)
print(multi_pivot)

print("\n" + "=" * 70)
print("MORE ADVANCED: Margins by Region (Profit / Sales)")
print("=" * 70)

# Create a new margin column first
sales_analysis = sales_data.copy()
sales_analysis['Margin%'] = (sales_analysis['Profit'] / sales_analysis['Sales'] * 100).round(1)

margin_pivot = sales_analysis.pivot_table(
    values='Margin%',
    index=['Region', 'Category'],
    columns='Quarter',
    aggfunc='mean'  # Average margin
)
print(margin_pivot)

print("\n💡 Business Insights from pivoted data:")
print("   • Compare performance across dimensions (Region, Category, Time)")
print("   • Spot trends (are margins improving quarter-to-quarter?)")
print("   • Identify gaps (which Region-Category had no data?)")


MULTI-LEVEL PIVOT: Complex reshaping for business dashboards

--- Dashboard View: Profit by Region & Category, quarters as columns ---
Quarter                Q1     Q2
Region Category                 
East   Clothing      2800   3200
       Electronics  11000  11600
       Food          1800   2000
North  Clothing      2400   3000
       Electronics   9000  10400
       Food          1600   1800
South  Clothing      3600   4400
       Electronics   7600   8200
       Food          2200   2600

MORE ADVANCED: Margins by Region (Profit / Sales)
Quarter               Q1    Q2
Region Category               
East   Clothing     20.0  20.0
       Electronics  20.0  20.0
       Food         20.0  20.0
North  Clothing     20.0  20.0
       Electronics  20.0  20.0
       Food         20.0  20.0
South  Clothing     20.0  20.0
       Electronics  20.0  20.0
       Food         20.0  20.0

💡 Business Insights from pivoted data:
   • Compare performance across dimensions (Region, Category, Time)
 

---

## Section 10: Real-World MultiIndex Example

In [58]:
# Build multi-level analysis from Superstore
if 'Region' in superstore.columns and 'Product Category' in superstore.columns and 'Sales' in superstore.columns:
    # Group by multiple dimensions
    regional_analysis = superstore.groupby(['Region', 'Product Category'])['Sales'].agg([
        'sum',
        'mean',
        'count'
    ]).round(2)
    
    print("Sales by Region and Category (MultiIndex):")
    print(regional_analysis)
    
    # Access specific level
    print("\nPivoted view:")
    pivoted = regional_analysis['sum'].unstack()
    print(pivoted)

Sales by Region and Category (MultiIndex):
                             sum    mean  count
Region Product Category                        
East   Electronics       1179.78  168.54      7
       Furniture         2672.32  267.23     10
       Office Supplies    628.70  209.57      3
North  Electronics       3178.88  211.93     15
       Furniture         3892.45  324.37     12
       Office Supplies   2027.66  202.77     10
South  Electronics       3044.90  234.22     13
       Furniture         2926.76  365.84      8
       Office Supplies   3397.71  308.88     11
West   Electronics       3351.94  304.72     11
       Furniture         2014.70  223.86      9
       Office Supplies   3448.15  313.47     11

Pivoted view:
Product Category  Electronics  Furniture  Office Supplies
Region                                                   
East                  1179.78    2672.32           628.70
North                 3178.88    3892.45          2027.66
South                 3044.90    2926.

---

## Section 11: Try It Yourself!

### Challenge 1: GroupBy Analysis - Netflix Content Strategy 🎬

**Business Question:** Which content type should we focus on for growth?

**Specific Tasks:**
1. Group Netflix data by 'type' (Movie vs TV Show)
2. Calculate: count, average release_year, and most common rating
3. Filter for types that have 50+ entries (significant volume)
4. Create a summary showing content strategy metrics

**Bonus:** Calculate the ratio of Movies to TV Shows to inform production planning

**Example output should show:**
```
type       count  avg_year  common_rating   pct_of_total
Movie      1500   2018      TV-MA           60%
TV Show    1000   2019      TV-14           40%
```

**Hint:** Use groupby().agg() with named aggregations, then filter() based on count

In [59]:
# Challenge 1: Your code here

# STEP 1: Group by 'type' and get counts
type_analysis = netflix.groupby('type').agg({
    'type': 'count',  # How many of each type?
    'release_year': 'mean',  # Average release year
    'rating': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'N/A'  # Most common rating
})

# STEP 2: Filter for types with 50+ entries
# significant_types = netflix.groupby('type').filter(...)

# STEP 3: Create final summary
# Your code here - show counts, percentages, and business insights

print("Type distribution in Netflix catalog:")
# Display results


Type distribution in Netflix catalog:


### Challenge 2: Merge & Join - Customer Order Analysis 📊

**Business Question:** Which customers have complete order history (ordered and had follow-up events)?

**Specific Tasks:**
1. Create 'new_customers' DataFrame with: CustomerID, Name, JoinDate
   - Use at least 5 customers
2. Create 'orders' DataFrame with: CustomerID, OrderID, Amount
   - Include some customers not in new_customers, some orders with no customer
3. Perform all 4 join types on these datasets
4. For each join, explain what business scenario it represents

**Example scenarios:**
- **INNER**: Customers with confirmed orders (quality data only)
- **LEFT**: All customers + their order history (or NULL if none)
- **RIGHT**: All orders + their customer info (find unmatched orders)
- **OUTER**: Complete reconciliation (find all anomalies)

**Analysis question:** Which join type reveals data quality issues? How would you use this for cleaning?

In [ ]:
# Challenge 2: Your code here

# STEP 1: Create sample customers dataframe
new_customers = pd.DataFrame({
    'CustomerID': [1, 2, 3, 4, 5],
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'JoinDate': ['2024-01-01', '2024-01-05', '2024-01-10', '2024-01-15', '2024-02-01']
})

# STEP 2: Create sample orders dataframe
orders = pd.DataFrame({
    'CustomerID': [1, 2, 1, 3, 6],  # Some overlap, some new (6 is not in customers)
    'OrderID': [101, 102, 103, 104, 105],
    'Amount': [150, 320, 210, 450, 89]
})

# STEP 3: Perform different joins and analyze results
# inner_result = pd.merge(new_customers, orders, on='CustomerID', how='inner')
# left_result = pd.merge(new_customers, orders, on='CustomerID', how='left')
# your code here for right and outer joins

# STEP 4: Analyze data quality
# What does each join type tell us about data quality?
print("Data quality analysis of each join type:")

Data quality analysis of each join type:


### Challenge 3: MultiIndex & Pivot - Regional Profitability Analysis 📈

**Business Question:** How do profit margins vary by region and product category across time?

**Specific Tasks:**
1. Using Superstore data, create a MultiIndex: [Region, Category]
2. Calculate metrics: Sum of Sales and Sum of Profit
3. Create a pivot table showing Profit% by Region (rows) and Category (columns)
4. Identify which Region-Category combo is most profitable
5. Use .xs() to get all data for the top-performing category

**Bonus Challenges:**
- Unstack to wide format and calculate row totals
- Add a "Total" row and column to your pivot table
- Rank regions by profitability and use .xs() to slice top 2 regions only
-  Transform sales data into % of each region's total

**Expected insights:**
- Visualize profit concentration (which combinations drive most profit?)
- Spot regional strengths and weaknesses
- Inform portfolio decisions (double down on winners, fix underperformers)

In [61]:
# Challenge 3: Your code here

if 'Sales' in superstore.columns and 'Profit' in superstore.columns:
    # STEP 1: Create MultiIndex
    # regional_data = superstore.set_index([...])
    
    # STEP 2: Calculate metrics by group
    # metrics = superstore.groupby([...]).agg({...})
    
    # STEP 3: Create profit margin pivot
    superstore_copy = superstore.copy()
    superstore_copy['Profit%'] = (superstore_copy['Profit'] / superstore_copy['Sales'] * 100).round(1)
    
    # profit_pivot = superstore_copy.pivot_table(
    #     values='Profit%',
    #     index=[...],
    #     columns=[...],
    #     aggfunc='mean'
    # )
    
    # STEP 4: Analysis
    # Which Region-Category is most profitable?
    # most_profitable_category = ...
    # Use xs() to slice that category
    
    print("Regional profitability analysis:")
    # Display results


Regional profitability analysis:


---

## Summary & Mastery Checklist

### ✅ Advanced Pandas Mastery

**GroupBy:**
- [ ] I understand split-apply-combine workflow
- [ ] I can group by single and multiple columns
- [ ] I use `.agg()` with multiple functions
- [ ] I can filter groups with `.filter()`
- [ ] I apply transforms within groups

**Merge:**
- [ ] I know the difference between join types
- [ ] I merge on columns and indices
- [ ] I handle duplicate column names with suffixes
- [ ] I understand when to use inner vs outer joins

**MultiIndex:**
- [ ] I create MultiIndex with `set_index()`
- [ ] I slice MultiIndex data with `.loc[]` and `.xs()`
- [ ] I pivot and unstack data
- [ ] I work with hierarchical data effectively

### Quick Reference - GroupBy

```python
# Count by group
df.groupby('col').size()

# Aggregate with multiple functions
df.groupby('col').agg({'col2': ['sum', 'mean']})

# Filter groups
df.groupby('col').filter(lambda x: len(x) > 10)

# Transform within groups
df.groupby('col')['col2'].transform(lambda x: (x - x.mean()) / x.std())
```

### Quick Reference - Merge

```python
# Inner join
pd.merge(df1, df2, on='key', how='inner')

# Left join
pd.merge(df1, df2, on='key', how='left')

# On index
pd.merge(df1, df2, left_index=True, right_index=True)
```

### Quick Reference - MultiIndex

```python
# Create MultiIndex
df.set_index(['col1', 'col2'])

# Access MultiIndex
df.loc['val1']
df.loc[('val1', 'val2')]
df.xs('val1', level='col1')

# Reshape
df.unstack()
df.stack()
```

---

## 🎓 Congratulations!

You've mastered **advanced Pandas techniques**! You can now:

✅ Aggregate and analyze complex grouped data  
✅ Combine datasets with different merge strategies  
✅ Work with hierarchical, multi-dimensional data  
✅ Reshape and restructure data for analysis  
✅ Solve real-world data engineering problems  

---

## Next Level Skills

**You're ready for:**
- Time series analysis
- Custom data transformations
- Real-world data pipelines
- Integration with machine learning
- Production data processing

**keep practicing and build amazing data projects!** 🚀